# 🛰️ SPLATSCOPE — Reconstruction vidéo → Gaussian Splat

Pipeline **gratuit** sur Google Colab : `vidéo → images → COLMAP → splatfacto (gsplat) → fichier .ply` prêt pour le viewer web.

**Avant de lancer :** menu `Exécution ▸ Modifier le type d'exécution ▸ T4 GPU`.

> ⚠️ Les installs sur Colab gratuit évoluent souvent. Si une cellule casse, redémarre l'exécution (`Exécution ▸ Redémarrer`) et relance depuis le début. La doc de référence reste **docs.nerf.studio**.


## 1 · Vérifier le GPU


In [ ]:
!nvidia-smi


## 2 · (Recommandé) Monter Google Drive
Colab est éphémère : monter le Drive te permet d'y déposer ta vidéo et d'y récupérer le résultat.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3 · Installer COLMAP, ffmpeg et Nerfstudio
Splatfacto n'utilise que **gsplat** (pas de tiny-cuda-nn), ce qui évite le plus gros piège d'install de Colab.
gsplat compile son code CUDA au premier entraînement — c'est normal que la cellule 6 mette quelques minutes la première fois.


In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq colmap ffmpeg
!pip install -q nerfstudio
print('\n✅ Installation terminée. Si des conflits de versions apparaissent, redémarre l’exécution et relance cette cellule.')


## 4 · Fournir la vidéo
Deux options — commente/décommente celle que tu utilises.

**Conseils de capture :** filme lentement, en faisant le tour du sujet, avec un bon recouvrement et un éclairage homogène. 20–60 s suffisent.


In [ ]:
# --- Option A : depuis Google Drive ---
# INPUT = '/content/drive/MyDrive/mes_captures/site.mp4'

# --- Option B : upload direct ---
from google.colab import files
up = files.upload()
INPUT = '/content/' + list(up.keys())[0]

print('Vidéo :', INPUT)


## 5 · Extraire les images + poses caméra (COLMAP)
`--num-frames-target` : plus d'images = plus précis mais plus lent. 100–150 est un bon compromis sur T4.


In [ ]:
!ns-process-data video \
  --data "$INPUT" \
  --output-dir /content/proc \
  --num-frames-target 150


## 6 · Entraîner le Gaussian Splat (splatfacto)
15 000 itérations suffisent pour une première passe (~15–30 min sur T4). Monte à 30 000 pour plus de qualité.

En cas de **mémoire GPU saturée (OOM)** : réduis `--num-frames-target` à l'étape 5, ou ajoute `--pipeline.model.cull_alpha_thresh 0.01`.


In [ ]:
!ns-train splatfacto \
  --data /content/proc \
  --max-num-iterations 15000 \
  --viewer.quit-on-train-completion True \
  --output-dir /content/outputs


## 7 · Exporter le fichier .ply web-ready


In [ ]:
import glob
cfg = sorted(glob.glob('/content/outputs/**/config.yml', recursive=True))[-1]
print('Config :', cfg)
!ns-export gaussian-splat --load-config {cfg} --output-dir /content/export
!ls -lh /content/export


## 8 · Récupérer le résultat
Le fichier `/content/export/splat.ply` se charge **directement** dans le viewer SPLATSCOPE.


In [ ]:
# Télécharger sur ta machine…
from google.colab import files
files.download('/content/export/splat.ply')

# …ou copier dans le Drive :
# !cp /content/export/splat.ply /content/drive/MyDrive/


---
### Et ensuite ?
1. **Alléger le fichier** (fortement conseillé) : ouvre `splat.ply` sur **superspl.at/editor**, supprime les *floaters*, recadre, puis exporte en `.splat` ou `.spz` compressé (jusqu'à −90 %).
2. **Publier** : dépose le fichier allégé dans ton repo (`models/`) si < ~50 Mo, sinon attache-le comme *asset* d'une **Release GitHub** et charge-le via son URL dans le viewer.
3. Ouvre ton appli GitHub Pages, bouton **Ouvrir un fichier** ou champ **URL** → ta scène s'affiche.
